# EDA
## Кредитный скоринг (Финтех школа) Tinkoff
### Описание файлов:
* `application_info.csv` - анкетные данные
* `default_flg.csv` - информация о дефолтах

### Описание полей:
* `id` - Анонимизированный идентификатор заявителя
* `application_dt` - Дата подачи заявки
* `sample_cd` - Категория выборки
* `education_cd` - Образование
* `gender_cd` - Пол
* `age` - Возраст
* `car_own_flg` - Флаг наличия автомобиля
* `car_type_flg` - Флаг наличия иномарки
* `appl_rej_cnt` - Количество отказанных прошлых заявок
* `good_work_flg` - Флаг наличия "хорошей" работы
* `Score_bki` - Скор балл по данным из бюро кредитных историй
* `out_request_cnt` - Количество запросов в бюро
* `region_rating` - Рейтинг региона
* `home_address_cd` - Категоризатор домашнего адреса
* `work_address_cd` - Категоризатор рабочего адреса
* `income` - Доход заявителя
* `SNA` - Связь заявителя с клиентами
* `first_time_cd` - Давность наличия информации о заявителе
* `Air_flg` - Наличие загран паспорта
* `default_flg` - Флаг дефолта по кредиту

***

**Ссылка на датасет: https://www.kaggle.com/c/fintech-credit-scoring/data**

***

Начнем исследование: импортируем необходимые библиотеки, прочитаем датасет с локального файла и выведем основную информацию.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from data_profiling import ProfileReport

df = pd.read_csv('./fintech-credit-scoring/application_info.csv')
target = pd.read_csv('./fintech-credit-scoring/default_flg.csv')

print("Размер df:", df.shape)
print("Размер target:", target.shape)
print("\nУникальных id в df:", df['id'].nunique())
print("Уникальных id в target:", target['id'].nunique())

Размер df: (205296, 19)
Размер target: (180816, 2)

Уникальных id в df: 205296
Уникальных id в target: 180816


Исходные данные содержат **205 296** анкет и **180 816** известных таргетов. Разница в **24 480** строк — это тестовая выборка, для которой исход кредита неизвестен.  
Учитывая, что датасет соревновательный, проверить работу модели на тестовых данных (без целевой переменной) у нас нет возможности, эти данные нам не понадобятся.  
Смело объединяем датасеты по номеру `id` и изучаем объединенный датасет.

In [2]:
merged = pd.merge(df, target, on='id')

In [3]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180816 entries, 0 to 180815
Data columns (total 20 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   id               180816 non-null  int64  
 1   application_dt   180816 non-null  object 
 2   sample_cd        180816 non-null  object 
 3   education_cd     180093 non-null  object 
 4   gender_cd        180816 non-null  object 
 5   age              180816 non-null  int64  
 6   car_own_flg      180816 non-null  object 
 7   car_type_flg     180816 non-null  object 
 8   appl_rej_cnt     180816 non-null  int64  
 9   good_work_flg    180816 non-null  int64  
 10  Score_bki        180816 non-null  float64
 11  out_request_cnt  180816 non-null  int64  
 12  region_rating    180816 non-null  int64  
 13  home_address_cd  180816 non-null  int64  
 14  work_address_cd  180816 non-null  int64  
 15  income           180816 non-null  int64  
 16  SNA              180816 non-null  int6

In [4]:
merged.head(15)

,id,application_dt,sample_cd,education_cd,gender_cd,age,car_own_flg,car_type_flg,appl_rej_cnt,good_work_flg,Score_bki,out_request_cnt,region_rating,home_address_cd,work_address_cd,income,SNA,first_time_cd,Air_flg,default_flg
0,1,01JAN2014,train,SCH,M,27,Y,Y,0,0,-1.917831,0,40,2,3,32000,1,4,N,0
1,2,01JAN2014,train,GRD,F,26,N,N,0,0,-1.153144,2,60,2,3,50000,2,1,N,0
2,3,01JAN2014,train,SCH,M,35,N,N,0,1,-1.732810,0,40,1,2,20000,4,3,N,0
3,4,01JAN2014,train,GRD,F,35,N,N,0,1,-2.552133,2,20,2,3,80000,1,3,N,0
4,5,01JAN2014,train,UGR,F,24,N,N,0,0,-1.914581,1,50,2,3,27000,1,2,N,0
5,6,01JAN2014,train,SCH,M,35,Y,Y,0,0,-1.920823,4,40,2,3,35000,4,3,N,0
6,7,01JAN2014,train,UGR,F,35,N,N,0,0,-1.990167,6,60,1,1,20000,1,3,N,0
7,8,01JAN2014,train,SCH,M,55,Y,N,0,0,-1.733450,1,40,2,3,8000,1,3,N,1
8,9,01JAN2014,train,GRD,M,34,Y,Y,0,0,-1.998422,2,50,1,1,30000,1,4,Y,0
9,10,01JAN2014,train,GRD,F,42,N,N,0,0,-1.949733,3,40,1,2,60000,3,3,N,0


Обратим внимание на типы данных и содержание столбцов:  
- `application_dt` – содержит информацию о дате, однако тип данных определен, как `object`, следует привести к `datetime`
- `sample_cd` – разделение на `training`, `validation` и `test` выборки, технический столбец, для наших целей не понадобится, удалим.
- `education_cd` – инфомрация об обазовании с ограниченном числом категорий, следует привести к `category`. Одновременно обратим внимание, что в данном столбце присутствуют пропуски, заполним `unknown`.
- `gender_cd` – пол клиента, приведем к `catogory`.
- `car_own_flg` и `car_type_flg` – данные о наличии у клиента автомобиля и его тип, признаки следует закодировать, также стоит обратить внимание, что в случае отсутствия автомобиля в таблице также будут отсутствовать данные о его типе, возможно, следует разбить комбанции признаков на отдельные бинарные флаги.
- `Score_bki` – по первым строкам датасета бросается в глаза распределение признака, отличающееся от привычного БКИ score (чем больше, тем лучше), вероятно, перед нами агрегированное значение некого risk-score параметра. Слеудет посмотреть распределение признака и его корреляцию с остальными признаками.
- `Air_flg` – бинарный классификатор, следует закодировать
- Отдельно стоит отметить, что большая часть данных представлена небольшими числами, даипозон `int64` представляется избыточным, для экономии памяти и ускорения работы, следует пройтись по целочисленным столбцам downcast-методом.  

Приступим.

In [5]:
# 1. Конвертация даты
merged['application_dt'] = pd.to_datetime(merged['application_dt'], format="%d%b%Y")

# 2. Удаление служебного столбца
merged = merged.drop(columns=['sample_cd'])

# 3. Обработка пропусков в образовании и перевод в категорию
merged['education_cd'] = merged['education_cd'].fillna('unknown').astype('category')
merged['gender_cd'] = merged['gender_cd'].astype('category')

# 4. Кодирование бинарных Y/N признаков
for col in ['car_own_flg', 'car_type_flg', 'Air_flg']:
    merged[col] = merged[col].map({'Y': 1, 'N': 0}).astype('int8')

# 5. Downcast для числовых признаков
for col in merged.select_dtypes(include=['int64']).columns:
    merged[col] = pd.to_numeric(merged[col], downcast='integer')

# 6. Приведем все названия столбков к нижнему регистру
merged.columns = merged.columns.str.lower()

In [6]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180816 entries, 0 to 180815
Data columns (total 19 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   id               180816 non-null  int32         
 1   application_dt   180816 non-null  datetime64[ns]
 2   education_cd     180816 non-null  category      
 3   gender_cd        180816 non-null  category      
 4   age              180816 non-null  int8          
 5   car_own_flg      180816 non-null  int8          
 6   car_type_flg     180816 non-null  int8          
 7   appl_rej_cnt     180816 non-null  int8          
 8   good_work_flg    180816 non-null  int8          
 9   score_bki        180816 non-null  float64       
 10  out_request_cnt  180816 non-null  int8          
 11  region_rating    180816 non-null  int8          
 12  home_address_cd  180816 non-null  int8          
 13  work_address_cd  180816 non-null  int8          
 14  income           180

In [7]:
merged.head(15)

,id,application_dt,education_cd,gender_cd,age,car_own_flg,car_type_flg,appl_rej_cnt,good_work_flg,score_bki,out_request_cnt,region_rating,home_address_cd,work_address_cd,income,sna,first_time_cd,air_flg,default_flg
0,1,2014-01-01,SCH,M,27,1,1,0,0,-1.917831,0,40,2,3,32000,1,4,0,0
1,2,2014-01-01,GRD,F,26,0,0,0,0,-1.153144,2,60,2,3,50000,2,1,0,0
2,3,2014-01-01,SCH,M,35,0,0,0,1,-1.732810,0,40,1,2,20000,4,3,0,0
3,4,2014-01-01,GRD,F,35,0,0,0,1,-2.552133,2,20,2,3,80000,1,3,0,0
4,5,2014-01-01,UGR,F,24,0,0,0,0,-1.914581,1,50,2,3,27000,1,2,0,0
5,6,2014-01-01,SCH,M,35,1,1,0,0,-1.920823,4,40,2,3,35000,4,3,0,0
6,7,2014-01-01,UGR,F,35,0,0,0,0,-1.990167,6,60,1,1,20000,1,3,0,0
7,8,2014-01-01,SCH,M,55,1,0,0,0,-1.733450,1,40,2,3,8000,1,3,0,1
8,9,2014-01-01,GRD,M,34,1,1,0,0,-1.998422,2,50,1,1,30000,1,4,1,0
9,10,2014-01-01,GRD,F,42,0,0,0,0,-1.949733,3,40,1,2,60000,3,3,0,0


Датасет успешно отформатирован и очищен, датасет небольшой, для первичного анализа взаимосвязей воспользуемся `ProfileReport` (ранее `PandasProfiler`).

In [8]:
profile = ProfileReport(merged, title="Pandas Profiling Report", explorative=True)
profile.to_file("Pandas_Profiling_Report_tinkoff.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 19/19 [00:00<00:00, 445.20it/s]
/Users/bocharov/Desktop/Обучение/HSE MSc/Year project/.venv/lib/python3.13/site-packages/data_profiling/model/pandas/discretize_pandas.py:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 0 0 ... 9 9 9]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  discretized_df.loc[:, column] = self._discretize_column(
/Users/bocharov/Desktop/Обучение/HSE MSc/Year project/.venv/lib/python3.13/site-packages/data_profiling/model/pandas/discretize_pandas.py:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[1 0 2 ... 0 1 4]' has dtype incompatible with int8, please explicitly cast to a compatible dtype first.
  discretized_df.loc[:, column] = self._discretize_column(
/Users/bocharov/Desktop/Обучение/HSE MSc/Year project/.venv/lib/python3.13/site-packages/data_profilin

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
# Построим матрицу сопряженности для уточнения взаимосвязи sna и first_time_cd

pd.crosstab(merged['first_time_cd'], merged['sna'], normalize='index')

sna,1,2,3,4
first_time_cd,,,,
1,0.191616,0.139535,0.089831,0.579018
2,0.495116,0.260125,0.114331,0.130428
3,0.708725,0.160880,0.049800,0.080594
4,0.900256,0.057893,0.011762,0.030090


In [10]:
merged['default_flg'].value_counts(normalize=True) # дисбаланс

default_flg
0    0.884778
1    0.115222
Name: proportion, dtype: float64

***

### Опишем найденные инсайты:

- Основное опасение было в сильной положительной связи `score_bki` и `default_flg` (целевой переменной), что могло бы стать проблемой для обучения модели, однако корреляция в `0.171` дает пространство для обучения модели и поиска нелинейных связей.
- `id` – технический признак, однако имеет корреляцию c `home_address`, `default_flg`, `score_bki`, в модели этот признак брать не просто бессмысленно, но и вредно из-за Data Leakage.
- `education_cd` – признак ординальный, уровень образования может быть закодирован.
- `car_own_flg` и `car_type_flg` имеют высокую корреляцию `0.688`. Это структурная (логическая) зависимость в данных: если у клиента нет автомобиля (`car_own_flg = 0`), то и тип автомобиля (`car_type_flg`) не определен. Признаки частично дублируют друг друга, так как описывают один и тот же объект собственности.
- `home_address_cd` и `work_address_cd` коррелируют на уровне `0.598`. Скорее всего, это артефакт категориального кодирования (например, близкие числовые коды обозначают похожие группы адресов), а не реальная географическая связь «живет и работает в одном районе». В данных признаки сильно пересекаются.
- Связка `income` и `region_rating` имеет корреляцию `0.330`. Это ожидаемая и логичная связь, отражающая социально-экономическую реальность датасета: чем выше рейтинг региона, тем выше средний доход клиента в этой выборке.
- `first_time_cd` и `sna` коррелируют на уровне `0.352`. Это указывает на то, что новые клиенты (`first_time_cd`) часто имеют специфический статус `sna`. Это может указывать на то, что клиенты, которые вперых обащаются в банк связаны с другими клиентами банка (говорит о рекомендациях, доверии).
- Связка `appl_rej_cnt` и `score_bki` (`0.180`), а также `age` с `income` (`-0.120`) показывает наличие интересных нелинейных паттернов в данных, которые не улавливаются простой линейной корреляцией. 
- Целевая переменная `default_flg` распределена неравномерно (`0.885` к `0.115`).

***